# NumPy Real Estate Market Analysis

Analyze property prices and characteristics with NumPy arrays, structured records, Boolean masks, broadcasting, sorting, and fancy indexing. The notebook uses the public [Housing Prices Regression dataset](https://www.kaggle.com/datasets/denkuznetz/housing-prices-regression).

The dataset describes `Square_Feet` as area in square meters, despite the field name. Therefore, the ratio in Task 4 is reported as price per listed area unit; it is not labeled as a conventional price per square foot.

## 1. Load and Explore the Data

The loader checks the notebook folder and common extraction folders first. If the CSV is not present, it downloads the public Kaggle ZIP into memory. No Kaggle credentials are required. If the download is unavailable, place `real_estate_dataset.csv` beside this notebook and rerun the cell.

In [1]:
from io import BytesIO
from pathlib import Path
from urllib.request import urlopen
from zipfile import ZipFile

import numpy as np

# Keep displayed records readable while retaining useful precision.
np.set_printoptions(precision=2, suppress=True)

csv_name = 'real_estate_dataset.csv'
dataset_url = 'https://www.kaggle.com/api/v1/datasets/download/denkuznetz/housing-prices-regression'
# Prefer a supplied CSV before downloading the public archive.
search_folders = [Path.cwd(), Path.cwd() / 'data', Path.cwd() / 'dataset',
                  Path.cwd() / 'datasets', Path.cwd() / 'archive']
csv_path = next((folder / csv_name for folder in search_folders
                 if (folder / csv_name).is_file()), None)

# Read local bytes or retrieve the CSV from inside Kaggle's ZIP archive.
if csv_path is not None:
    csv_bytes = csv_path.read_bytes()
    print(f'Loaded local dataset: {csv_path}')
else:
    print('Local CSV not found; downloading the public Kaggle dataset...')
    with urlopen(dataset_url, timeout=30) as response:
        archive_bytes = response.read()
    with ZipFile(BytesIO(archive_bytes)) as archive:
        csv_bytes = archive.read(csv_name)
    print(f'Loaded {csv_name} from the Kaggle ZIP (in memory).')

# Parse the header into named fields so columns can be accessed by name.
properties = np.genfromtxt(
    BytesIO(csv_bytes), delimiter=',', names=True, dtype=None, encoding='utf-8-sig'
)
properties = np.atleast_1d(properties)
# Catch an incompatible CSV before later analysis tries to access missing fields.
required_fields = {'ID', 'Square_Feet', 'Num_Bedrooms', 'Location_Score', 'Price'}
missing_fields = required_fields.difference(properties.dtype.names or ())
if missing_fields:
    raise ValueError(f'Dataset is missing required fields: {sorted(missing_fields)}')

print(f'Array shape: {properties.shape}')
print(f'Fields: {properties.dtype.names}')
print(f'Data types: {properties.dtype}')
print('First five records:')
print(properties[:5])

Local CSV not found; downloading the public Kaggle dataset...


Loaded real_estate_dataset.csv from the Kaggle ZIP (in memory).
Array shape: (500,)
Fields: ('ID', 'Square_Feet', 'Num_Bedrooms', 'Num_Bathrooms', 'Num_Floors', 'Year_Built', 'Has_Garden', 'Has_Pool', 'Garage_Size', 'Location_Score', 'Distance_to_Center', 'Price')
Data types: [('ID', '<i8'), ('Square_Feet', '<f8'), ('Num_Bedrooms', '<i8'), ('Num_Bathrooms', '<i8'), ('Num_Floors', '<i8'), ('Year_Built', '<i8'), ('Has_Garden', '<i8'), ('Has_Pool', '<i8'), ('Garage_Size', '<i8'), ('Location_Score', '<f8'), ('Distance_to_Center', '<f8'), ('Price', '<f8')]
First five records:
[(1, 143.64, 1, 3, 3, 1967, 1, 1, 48, 8.3 ,  5.94, 602134.82)
 (2, 287.68, 1, 2, 1, 1949, 0, 1, 37, 6.06, 10.83, 591425.14)
 (3, 233.  , 1, 3, 2, 1923, 1, 0, 14, 2.91,  6.9 , 464478.7 )
 (4, 199.66, 5, 2, 2, 1918, 0, 0, 17, 2.07,  8.28, 583105.66)
 (5,  89.  , 4, 3, 3, 1999, 1, 0, 34, 1.52, 14.65, 619879.14)]


## 2. Analyze Property Prices

Calculate central tendency, range, and selected price percentiles to describe the market.

In [2]:
# Select the price field from the structured array and summarize its distribution.
prices = properties['Price']
price_percentiles = np.percentile(prices, [25, 50, 75, 90, 95])

print(f'Mean price:   ${np.mean(prices):,.2f}')
print(f'Median price: ${np.median(prices):,.2f}')
print(f'Minimum:      ${np.min(prices):,.2f}')
print(f'Maximum:      ${np.max(prices):,.2f}')
print('Percentiles (25th, 50th, 75th, 90th, 95th):')
print(np.round(price_percentiles, 2))
print('Interpretation: The mean and median summarize a typical price, while the wide minimum-to-maximum range shows substantial variation across properties.')

Mean price:   $582,209.63
Median price: $574,724.11
Minimum:      $276,892.47
Maximum:      $960,678.27
Percentiles (25th, 50th, 75th, 90th, 95th):
[503080.34 574724.11 665942.3  743808.06 792131.55]
Interpretation: The mean and median summarize a typical price, while the wide minimum-to-maximum range shows substantial variation across properties.


## 3. Analyze Property Size

Compare the average and median area and inspect the smallest and largest properties.

In [3]:
# Extract property area once so later size comparisons use the same values.
sizes = properties['Square_Feet']
average_size = np.mean(sizes)
print(f'Mean size:   {average_size:,.2f} listed area units')
print(f'Median size: {np.median(sizes):,.2f} listed area units')
print(f'Minimum:     {np.min(sizes):,.2f} listed area units')
print(f'Maximum:     {np.max(sizes):,.2f} listed area units')
print('Interpretation: The mean and median are relatively close, while the min-to-max range indicates that the dataset contains both compact and much larger properties.')

Mean size:   174.64 listed area units
Median size: 178.29 listed area units
Minimum:     51.27 listed area units
Maximum:     298.24 listed area units
Interpretation: The mean and median are relatively close, while the min-to-max range indicates that the dataset contains both compact and much larger properties.


## 4. Calculate Price per Area Unit

Element-wise division pairs each property's price with its own area. The Kaggle description says `Square_Feet` is measured in square meters, so this is price per reported area unit; convert the area first if a true price-per-square-foot figure is required.

In [4]:
# Divide each property's price by its matching area using element-wise operations.
price_per_area_unit = prices / sizes
print(f'Mean:   {np.mean(price_per_area_unit):,.2f} price units per listed area unit')
print(f'Median: {np.median(price_per_area_unit):,.2f}')
print(f'Minimum:{np.min(price_per_area_unit):,.2f}')
print(f'Maximum:{np.max(price_per_area_unit):,.2f}')
print('Interpretation: The mean exceeding the median indicates that some properties have especially high price-to-area ratios; these may reflect attributes beyond size.')

Mean:   4,008.82 price units per listed area unit
Median: 3,361.19
Minimum:1,636.48
Maximum:11,572.99
Interpretation: The mean exceeding the median indicates that some properties have especially high price-to-area ratios; these may reflect attributes beyond size.


## 5. Identify Top-Quartile Properties

Use the 75th percentile as the cutoff for the most expensive quarter of the dataset.

In [5]:
# Use the 75th percentile as the price threshold for the top quarter.
top_quartile_cutoff = np.percentile(prices, 75)
top_price_mask = prices >= top_quartile_cutoff
# Apply the Boolean mask to retain only top-quartile records.
top_quartile = properties[top_price_mask]
print(f'75th-percentile price cutoff: ${top_quartile_cutoff:,.2f}')
print(f'Number of properties: {np.count_nonzero(top_price_mask)} of {len(properties)}')
print(f'Top-quartile average price: ${np.mean(top_quartile["Price"]):,.2f}')
print(f'Overall average price:      ${np.mean(prices):,.2f}')
print(f'Difference from overall average: ${np.mean(top_quartile["Price"]) - np.mean(prices):,.2f}')

75th-percentile price cutoff: $665,942.30
Number of properties: 125 of 500
Top-quartile average price: $744,142.53
Overall average price:      $582,209.63
Difference from overall average: $161,932.90


## 6. Apply Multiple Conditions

Filter for properties larger than the dataset average, in the top price quartile, and with a location score of at least 8. All three Boolean conditions must be true for a record to qualify.

In [6]:
location_scores = properties['Location_Score']
# Combine the criteria element-wise; a record must satisfy every condition.
combined_mask = (sizes > average_size) & top_price_mask & (location_scores >= 8)
matching_properties = properties[combined_mask]
print(f'Properties meeting all conditions: {len(matching_properties)}')
print('Matching records (ID, size, bedrooms, location score, price):')
print(matching_properties[['ID', 'Square_Feet', 'Num_Bedrooms', 'Location_Score', 'Price']])
if len(matching_properties):
    print(f'Average price: ${np.mean(matching_properties["Price"]):,.2f}')
    print(f'Average size: {np.mean(matching_properties["Square_Feet"]):,.2f} listed area units')
    print(f'Average location score: {np.mean(matching_properties["Location_Score"]):.2f}')

Properties meeting all conditions: 17
Matching records (ID, size, bedrooms, location score, price):
[(  8, 266.54, 5, 9.37, 875352.55) ( 63, 257.18, 4, 9.95, 695024.31)
 (113, 282.42, 3, 9.01, 787131.57) (127, 254.5 , 3, 9.92, 743412.14)
 (151, 277.07, 5, 9.04, 960678.27) (201, 210.51, 5, 8.18, 885909.22)
 (273, 266.77, 5, 9.52, 789119.86) (314, 290.66, 4, 9.9 , 789822.93)
 (323, 215.05, 5, 9.79, 747398.28) (383, 207.24, 5, 8.07, 845911.37)
 (384, 269.37, 5, 8.61, 676685.28) (397, 264.6 , 5, 9.12, 842272.54)
 (426, 187.38, 5, 9.15, 669863.66) (463, 236.51, 4, 9.25, 702645.57)
 (470, 290.81, 3, 8.28, 708362.29) (481, 207.35, 3, 8.29, 695642.11)
 (485, 196.08, 5, 8.84, 723834.85)]
Average price: $772,886.28
Average size: 245.88 listed area units
Average location score: 9.08


## 7. Compare Prices to the Average with Broadcasting

NumPy broadcasts the single overall-average scalar across the price array during comparison, producing one Boolean result per property.

In [7]:
overall_average_price = np.mean(prices)
# Comparing an array with a scalar broadcasts the average to every price.
above_average_mask = prices > overall_average_price
below_average_mask = prices < overall_average_price
equal_to_average_mask = prices == overall_average_price
print(f'Overall average price: ${overall_average_price:,.2f}')
print(f'Above average: {np.count_nonzero(above_average_mask)} properties')
print(f'Below average: {np.count_nonzero(below_average_mask)} properties')
print(f'Exactly average: {np.count_nonzero(equal_to_average_mask)} properties')
# Use each mask to list example property IDs from both groups.
print('First ten above-average property IDs:', properties['ID'][above_average_mask][:10])
print('First ten below-average property IDs:', properties['ID'][below_average_mask][:10])

Overall average price: $582,209.63
Above average: 240 properties
Below average: 260 properties
Exactly average: 0 properties
First ten above-average property IDs: [ 1  2  4  5  6  8  9 12 13 14]
First ten below-average property IDs: [ 3  7 10 11 15 16 17 18 19 22]


## 8. Find the 10 Most Expensive Properties

Sort price indices from highest to lowest, then use fancy indexing to retrieve the corresponding structured records and requested fields.

In [8]:
# Sort the indices, then reverse the final ten so prices are highest first.
sorted_indices = np.argsort(prices)
top_ten_indices = sorted_indices[-10:][::-1]
# Fancy indexing retrieves complete records at the selected positions.
top_ten = properties[top_ten_indices]
top_ten_fields = top_ten[['ID', 'Square_Feet', 'Num_Bedrooms', 'Location_Score', 'Price']]
print('Top 10 properties, ordered by price (descending):')
print(top_ten_fields)

Top 10 properties, ordered by price (descending):
[(151, 277.07, 5, 9.04, 960678.27) ( 53, 284.87, 5, 7.61, 909199.56)
 (201, 210.51, 5, 8.18, 885909.22) (  8, 266.54, 5, 9.37, 875352.55)
 (474, 272.79, 5, 6.32, 867558.3 ) (500, 296.55, 4, 7.89, 864299.5 )
 (108, 238.89, 5, 2.54, 860013.26) (368, 276.34, 4, 3.86, 857487.18)
 (379, 253.62, 5, 7.43, 854374.52) ( 70, 296.72, 4, 4.07, 854304.13)]


## 9. Final Analysis and Business Insight

Compare average size, bedroom count, and location score for the ten most expensive properties against the full dataset. The closing insight is generated from these values.

In [9]:
# Compare the top ten's average characteristics with the full dataset.
top_ten_mean_size = np.mean(top_ten['Square_Feet'])
overall_mean_size = np.mean(properties['Square_Feet'])
top_ten_mean_bedrooms = np.mean(top_ten['Num_Bedrooms'])
overall_mean_bedrooms = np.mean(properties['Num_Bedrooms'])
top_ten_mean_location = np.mean(top_ten['Location_Score'])
overall_mean_location = np.mean(properties['Location_Score'])

print(f'Average size: top 10 = {top_ten_mean_size:.2f}; all properties = {overall_mean_size:.2f}')
print(f'Average bedrooms: top 10 = {top_ten_mean_bedrooms:.2f}; all properties = {overall_mean_bedrooms:.2f}')
print(f'Average location score: top 10 = {top_ten_mean_location:.2f}; all properties = {overall_mean_location:.2f}')
print()
# Phrase the insight based on the measured comparisons rather than assumptions.
size_comparison = 'larger' if top_ten_mean_size > overall_mean_size else 'smaller'
bedroom_comparison = 'more' if top_ten_mean_bedrooms > overall_mean_bedrooms else 'fewer'
location_comparison = 'higher' if top_ten_mean_location > overall_mean_location else 'lower'
print(
    f'Business insight: The 10 most expensive properties are {size_comparison} on average '
    f'({top_ten_mean_size:.1f} versus {overall_mean_size:.1f} listed area units) and have '
    f'{bedroom_comparison} bedrooms ({top_ten_mean_bedrooms:.1f} versus {overall_mean_bedrooms:.1f}). '
    f'Their average location score is {location_comparison} '
    f'({top_ten_mean_location:.2f} versus {overall_mean_location:.2f}). '
    'In this dataset, larger homes, more bedrooms, and stronger locations tend to accompany higher prices, '
    'but these are descriptive patterns from only ten top-priced properties and do not establish causation.'
)

Average size: top 10 = 267.39; all properties = 174.64
Average bedrooms: top 10 = 4.70; all properties = 2.96
Average location score: top 10 = 6.63; all properties = 5.16

Business insight: The 10 most expensive properties are larger on average (267.4 versus 174.6 listed area units) and have more bedrooms (4.7 versus 3.0). Their average location score is higher (6.63 versus 5.16). In this dataset, larger homes, more bedrooms, and stronger locations tend to accompany higher prices, but these are descriptive patterns from only ten top-priced properties and do not establish causation.
